# 第15回：Show & Tellと自社データへの橋渡し

**今日の問い：自社データで始めるなら、最初の小さな一歩は何か。**

**セルの動かし方**：各セル（灰色の枠）を選んで `Shift + Enter`（またはセル左の▷ボタン）を押すと実行できます。
**上から順に**実行してください。前のセルを飛ばすと、後のセルでエラーになります。

`TRY`は全員、`CHANGE`は値を1つ変える練習、`CHALLENGE`は余裕がある人向けです。
`DEEP DIVE`・`APPENDIX`は経験者や自習向けの発展で、飛ばしても本編は完結します。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
# 【準備セル】教材フォルダの場所を自動で見つけます。中身は今は理解しなくてOK、そのまま実行してください。
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- モデルの目的・検証・結果・限界を短く説明し、再現可能に共有する
- 学習済みPipelineをjoblibで保存し、モデルカードを関数で生成する
- 適用領域と較正の観点から、使ってよい範囲と監視項目を決める

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- モデルカード：用途・データ・評価・限界をまとめた記録
- 適用領域：モデルを使ってよい対象と条件
- 永続化：学習済みモデルをファイルへ保存すること
- ドリフト：運用後に入力や関係が変わること
- 監視：運用後の入力や性能変化を確認すること

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


## 最終回：成果を「伝え」、自社データへ「橋渡し」する

最後は、作ったものを人に伝え、次の一歩へつなぐ回です。データサイエンスは「良いモデルを作って終わり」
ではなく、**「使われて初めて価値になる」**。ここまで学んだことを、発表と持ち帰りの形にまとめます。

まず**再現性**の確認から。第14回のNotebookを`Kernel`→`Restart Kernel and Run All Cells`で頭から実行し、
同じ提出CSVができることを確かめます（第12回の再現性の実践）。次のセルは、発表で共有できる基本の数字を出します。


In [ ]:
import pandas as pd
experiment_data = pd.read_csv(DATA / "compound_experiments.csv")
print("共有する候補")
print("データ件数:", len(experiment_data))
print("活性率:", round(experiment_data["active"].mean(), 3))
print("収率の中央値:", experiment_data["yield_pct"].median())


### 読みどころ

こうした基本統計（件数・活性率・中央値）は、発表の最初に置くと聞き手が状況をつかめます。**派手な
モデルより、まずデータの素性を1〜2行で言える**ことが、信頼される発表の土台です。


## 1人5分のShow & Tell

次のうち1つを選んで共有します：**面白かった図 / 改善した実験 / 悪化したが学びがあった実験 /
Copilotへの良かった聞き方 / 自社テーマへ持ち帰りたい考え方**。

完成度は競いません。むしろ**「悪化したが学びがあった実験」**の共有が、チーム全体の学びになります
（うまくいかない筋を先に潰せる）。「1回の高スコア」より「再現できる気づき」を持ち寄ります。


## 自社テーマ1枚シート

この教材の集大成として、自分のテーマを1枚に落とします。機密情報や実データは書かず、一般化した
表現で。**第5回の問題設定がここに戻ってきます**——予測時点と使えない情報を、もう一度自分の言葉で。

| 項目 | 記入内容 |
|---|---|
| 利用者と判断 | 誰が何を決めるか |
| 予測時点 | いつ予測するか |
| 目的変数 | 何を予測するか |
| 説明変数候補 | その時点で得られる情報 |
| 使えない情報 | 未来情報、測定後情報、機密上使えない情報 |
| 評価方法 | 指標と分割単位 |
| 単純な基準 | 平均、最頻値、現在の判断方法など |
| 最初の実験 | 1〜2週間で試せる小さな範囲 |

この1枚が、勉強会後に自社データで踏み出す**最初の一歩の設計図**になります。


## DEEP DIVE：発表で終わらせない——再現・共有・安全な運用

発展として、実務で「モデルを渡す」ときに必要な3つを扱います。**永続化**（保存して再利用）、
**モデルカード**（使い方の説明書）、**適用領域**（予測してよい範囲）。どれも「モデルを安全に使ってもらう」
ための工夫です。


### 永続化：学習済みモデルをファイルに保存する

毎回学習し直すのは非効率で、再現性も損なわれます。`joblib`で学習済みPipelineを**丸ごと保存**し、
読み直しても**同じ予測**になることを`assert`で確かめます。前処理も一緒に保存される点が重要です。


In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

data = pd.read_csv(DATA / "compound_experiments.csv")
feat = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X_tr, X_te, y_tr, y_te = train_test_split(data[feat], data["active"], test_size=0.25, random_state=42, stratify=data["active"])
final = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)).fit(X_tr, y_tr)
path = ROOT / "workspace" / "final_model.joblib"
joblib.dump(final, path)
reloaded = joblib.load(path)
assert np.array_equal(final.predict(X_te), reloaded.predict(X_te)), "保存前後で予測が一致しません"
print("保存し読み直しても同じ予測:", path)


### 読みどころ

`assert`が通り「同じ予測」と出れば、保存→配布→再利用の流れが安全に回ることの確認になります。
`Pipeline`ごと保存するので、**受け取った人は前処理を意識せず`predict`するだけ**。第9回でPipelineに
まとめた恩恵がここで効きます。


### モデルカード：使い方の説明書を関数で作る

モデルは「精度の数字」だけ渡してもトラブルの元です。**誰向けか・何を決めるためか・限界・禁止事項**を
1枚にまとめた**モデルカード**を、関数で自動生成します。第5回の問題設定が、そのまま説明書になります。


In [ ]:
from sklearn.metrics import f1_score

def build_model_card(name, estimator, X_valid, y_valid, notes) -> pd.DataFrame:
    "モデルの用途と評価をまとめた1枚のカードを作る。"
    pred = estimator.predict(X_valid)
    items = {
        "モデル名": name,
        "検証F1": round(f1_score(y_valid, pred), 3),
        "想定利用者": notes["利用者"],
        "支援する判断": notes["判断"],
        "既知の限界": notes["限界"],
        "使ってはいけない条件": notes["禁止"],
    }
    return pd.DataFrame({"項目": list(items), "内容": list(items.values())})

build_model_card("活性スクリーナ", reloaded, X_te, y_te, {
    "利用者": "実験担当者", "判断": "追試する候補の優先順位",
    "限界": "新規scaffoldでは精度低下の可能性", "禁止": "測定後の列を入力に使うこと",
})


### 読みどころ

出来上がったカードには、性能（F1）と**使う上での注意**が並びます。特に「使ってはいけない条件（測定後の
列を入力にしない）」は、第5〜6回のリークの教訓そのもの。**精度より先に限界を書く**のが、信頼される
モデル提供者の作法です。


### 適用領域：予測してよい範囲を数値化する

モデルは、学習データと似た試料には強いですが、かけ離れた試料では当てになりません。学習データからの
**近傍距離**を測り、遠すぎる（範囲外の）試料を「要確認」に自動で仕分けます。95%点を閾値にします。


In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

train_filled = X_tr.fillna(X_tr.median())
scaler = StandardScaler().fit(train_filled)
nn = NearestNeighbors(n_neighbors=5).fit(scaler.transform(train_filled))
train_dist = nn.kneighbors(scaler.transform(train_filled))[0].mean(axis=1)
threshold = np.quantile(train_dist, 0.95)
valid_dist = nn.kneighbors(scaler.transform(X_te.fillna(X_tr.median())))[0].mean(axis=1)
out_of_domain = valid_dist > threshold
print(f"適用領域外と判定された検証試料: {int(out_of_domain.sum())} / {len(valid_dist)} 件")
print("範囲外は予測を鵜呑みにせず、要確認に回す運用が考えられる。")


### 読みどころ、そして全15回のまとめ

範囲外と判定された試料は、予測を鵜呑みにせず人が確認する——これが**安全にAIを使う**ということです。

全15回を貫いた芯は1つ：**「良いスコア」ではなく「意味のある予測」**。予測時点を決め、ベースラインと比べ、
リークを避け、正しく評価し、1つずつ改善を記録し、限界とともに伝える。この習慣こそが、皆さんが自社
データへ持ち帰るいちばんの財産です。お疲れさまでした。


## APPENDIX（任意・追加演習）

「渡せる成果物」を実際に書き出します。90分の外の自習向けです。まず**モデルカードをMarkdown＋JSONで
保存**し、第三者が読める形にします。


In [ ]:
import json

card = build_model_card("活性スクリーナ", reloaded, X_te, y_te, {
    "利用者": "実験担当者", "判断": "追試候補の優先順位",
    "限界": "新規scaffoldで精度低下の可能性", "禁止": "測定後の列を入力に使うこと",
})
lines = ["# モデルカード", ""]
for _, r in card.iterrows():
    lines.append(f"- **{r['項目']}**: {r['内容']}")
(ROOT / "workspace" / "model_card.md").write_text("\n".join(lines), encoding="utf-8")

meta = {"features": feat, "n_train": int(len(X_tr)), "model": "RandomForest(max_depth=5)"}
(ROOT / "workspace" / "model_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print("保存: workspace/model_card.md, workspace/model_meta.json")
print("\n".join(lines))


### 出力の読み方

`model_card.md`は人が読む説明書、`model_meta.json`は機械が読む来歴（使った特徴量・学習件数・モデル種別）。
モデルと一緒にこの2つを残すと、**半年後の自分や引き継ぎ先が再現・判断できます**。


### ドリフトを模擬する：入力がずれたら「監視」で気づけるか

運用後、測定装置のずれなどで入力分布が変わる（ドリフト）ことがあります。テストの温度を+20℃ずらし、
**正解ラベルが無くても異常に気づけるか**を確かめます。運用中は正解（活性の実測）がすぐには手に入らない
ため、F1のような指標は即座には測れません。だからこそ、正解なしで検知できる監視が重要になります。


In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

drift = X_te.copy()
drift["temperature_c"] = drift["temperature_c"] + 20

# (1) 正解が無くても分かる変化：予測の陽性率
print(f"予測の陽性率: {reloaded.predict(X_te).mean():.3f} → {reloaded.predict(drift).mean():.3f}")

# (2) 監視：元データとドリフト後を見分けられるか（adversarial validation, 第6回）
cols = X_te.columns.tolist()
combined = pd.concat([X_te.assign(is_drift=0), drift.assign(is_drift=1)], ignore_index=True)
filled = combined[cols].fillna(combined[cols].median())
auc = cross_val_score(RandomForestClassifier(n_estimators=200, random_state=42), filled, combined["is_drift"], cv=5, scoring="roc_auc").mean()
print(f"監視AUC: {auc:.3f}（0.5=変化なし / 1.0に近い=明確な分布変化）")

# 参考：正解が手に入ればF1でも確認できる（運用中は正解が遅れて届く）
print(f"参考F1: {f1_score(y_te, reloaded.predict(X_te)):.3f} → {f1_score(y_te, reloaded.predict(drift)):.3f}")


### 出力の読み方

- **監視AUCが0.5をはっきり上回る**なら、元データとドリフト後をモデルが見分けられる＝入力分布が変化した、という警報です。温度を+20℃ずらしたので、AUCは0.5より明確に高く出るはずです（1に近いほど変化が大きい）。
- **予測の陽性率**の変化も、正解ラベル無しで「何かが変わった」と気づける手がかりです。
- 一方、**参考F1は運用中すぐには測れません**（正解が遅れて届くため）。しかもこのデータ・特徴量では変化が小さく、性能指標だけに頼ると見逃しかねません。だからこそ、正解なしで異常を検知するadversarial validation（第6回）のような監視が実務で効きます。


### 成績表をファイルに書き出す

`classification_report`を表として保存します。発表資料や引き継ぎに添付できる、機械可読な成績表です。


In [ ]:
from sklearn.metrics import classification_report

rep = classification_report(y_te, reloaded.predict(X_te), target_names=["非活性", "活性"], output_dict=True)
rep_df = pd.DataFrame(rep).T.round(3)
rep_df.to_csv(ROOT / "workspace" / "classification_report.csv")
display(rep_df)


### 出力の読み方、そしてこの教材の終わりに

クラスごとのprecision/recall/F1と全体のaccuracyが表になり、CSVで保存されます。数字だけを渡すのではなく、
**モデルカード（用途と限界）＋メタ情報（来歴）＋成績表**をひとまとめに渡す——ここまでできれば、
「作って終わり」から「使ってもらえる」への橋を渡せています。全15回、おつかれさまでした。


## よくある誤り

- スコアだけを成果として示す
- 自社データの利用許可や来歴を省略する
- 本番投入を最初の試行にする

## SELF-STUDY（任意・30〜60分）

- 保存したPipelineを読み直し、同じ入力で同じ予測になるか検証する
- 適用領域スコアを閾値化し、範囲外の試料を要確認として仕分ける

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. このモデルは誰の何の判断を助けるか
2. 適用領域をどう数値化したか
3. 運用後に監視すべき指標は何か

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
